# Phase 03 — Step 14
## Live Evidence-Grounded OpenAI Recommendation System

Notebook 10 already designed a deterministic, evidence-grounded recommendation engine:

1. frozen ElasticNet model predicts employability
2. construct scores, item scores, and local SHAP become evidence
3. practical actions come from the verified item library

Notebook 10 did **not** call OpenAI. This notebook documents the live service upgrade that now uses the same Phase 01/02 pattern:

- the frozen ElasticNet model still produces `EVS_avg`
- OpenAI only converts validated model evidence into personalized guidance
- a rule-based fallback is used if the API is unavailable

The OpenAI key is loaded only from `samindi/Employability/.env`.

Notebooks 01–12 remain the original 1000-row development evidence. Phase 01 and Phase 02 services are not modified here.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "phase3_psycap":
    PROJECT_DIR = PROJECT_DIR.parent.parent

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from employability_service import build_phase_services

services = build_phase_services(PROJECT_DIR)
phase3 = services["phase3"]
print("Phase:", phase3.phase_label)
print("Model loaded:", phase3.exists())
print("Model type:", phase3.model_type)

## Sample PsyCap profile

This mixed profile has stronger self-efficacy items and developing hope/optimism items, matching the notebook 10 test style.

In [ ]:
sample_responses = {
    "PsyCap1": 3, "PsyCap2": 4, "PsyCap3": 3,
    "PsyCap4": 3, "PsyCap5": 4, "PsyCap6": 3,
    "PsyCap7": 4, "PsyCap8": 3, "PsyCap9": 4,
    "PsyCap10": 5, "PsyCap11": 4, "PsyCap12": 4,
}

result = phase3.predict(sample_responses)

print("Predicted EVS_avg:", result["prediction"]["evs_avg"])
print("Recommendation source:", result.get("recommendation_source"))
print("Construct scores:", result["construct_scores"])
print("SHAP:", result.get("shap_explanation"))

In [ ]:
guidance = result.get("ai_recommendation", {})

print(guidance.get("overall_career_readiness_summary", ""))
print("\nTop strengths:")
for item in guidance.get("top_3_strengths", []):
    print(" -", item)
print("\nPriority gaps:")
for item in guidance.get("top_3_areas_for_improvement", []):
    print(" -", item)

## Evidence contract

- Score source: frozen ElasticNet + StandardScaler
- Features: `Hope_avg`, `Optimism_avg`, `Resilience_avg`, `SelfEfficacy_avg`
- Explanation: Linear SHAP on the scaled constructs
- Item actions: notebook 10 `ACTION_LIBRARY`
- Guidance: OpenAI JSON schema, with notebook 10 rule-based fallback
- Live module: `phase3_openai_guidance.py`